In [1]:
import yfinance as yf
import pandas as pd
import numpy as np


In [2]:
# sk하이닉스의 데이터를 수집
hynix = yf.Ticker('000660.KS')
# 최근 2년간의 데이터를 추출
df = hynix.history(period = '2y')
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-05-20 00:00:00+09:00,189403.297902,190289.740794,185759.032680,187236.437500,3592128,0.0,0.0
2024-05-21 00:00:00+09:00,190486.739372,190585.233032,188319.878841,189107.828125,2468109,0.0,0.0
2024-05-22 00:00:00+09:00,189009.321217,194721.953125,188024.384682,194721.953125,3949439,0.0,0.0
2024-05-23 00:00:00+09:00,200434.574570,200927.042813,195214.411203,196987.296875,4681849,0.0,0.0
2024-05-24 00:00:00+09:00,196790.330155,199449.658941,194524.976004,195608.406250,3793872,0.0,0.0


In [3]:
# 종가, 거래량 데이터만 사용
df = df [['Close','Volume']]

In [4]:
# 기술적 지표 : 20일간의 이동평균선
# 20개의 데이터를 묶어준다. -> rolling(n)
df['MA20'] = df['Close'].rolling(20).mean()
df.iloc[15 : 25, ]

,Close,Volume,MA20
Date,,,
2024-06-11 00:00:00+09:00,209299.015625,3070163,NaN
2024-06-12 00:00:00+09:00,211761.359375,2151331,NaN
2024-06-13 00:00:00+09:00,218655.906250,5777279,NaN
2024-06-14 00:00:00+09:00,217670.984375,3311223,NaN
2024-06-17 00:00:00+09:00,219640.843750,2198340,199942.117188
2024-06-18 00:00:00+09:00,230967.625000,3136231,202128.676563
2024-06-19 00:00:00+09:00,229982.703125,3758652,204172.420313
2024-06-20 00:00:00+09:00,233922.437500,2927358,206132.444531
2024-06-21 00:00:00+09:00,230475.156250,3848394,207806.837500


In [5]:
# 타겟 변수 -> 내일의 종가가 내일의 20일 평균선보다 높은가? (1: 상승 돌파, 0: 하락)
# 인덱스를 한칸씩 위로 올린다. shift(1) -> 1칸씩 내린다, shift(-1) -> -1칸씩 이동(위로 1칸 이동)
df['target'] = (df['Close'].shift(-1) > df['MA20'].shift(-1)).astype(int)

- astype() -> 타입을 변경하는 함수 ( 문자형 데이터를 숫자형으로 변경할때는 int)
    - 컬럼 내의 데이터가 '1' -> 1
    - 'a' -> error 발생
- to_numeric() -> 범주형 데이터를 수치형으로 변경(문자로 되어있는 숫자를 숫자 타입으로 변경)
    - '1' -> 1
    - '-' -> NaN

In [6]:
df['target'].value_counts()

target
1    295
0    190
Name: count, dtype: int64

In [8]:
# MLP 감성 점수 데이터를 추가
# 원래는 기사를 크롤링하여 감성 점수를 예측하고 입력해야하지만
# 무작위한 데이터를 입력
df['MLP_Sentiment'] = np.random.uniform(-1, 1, len(df))

In [10]:
df['MLP_Sentiment'].describe()

count    485.000000
mean      -0.042259
std        0.569344
min       -0.998794
25%       -0.536119
50%       -0.061772
75%        0.429981
max        0.997912
Name: MLP_Sentiment, dtype: float64

In [ ]:
# !pip install OpenDartReader

INFO: pip is looking at multiple versions of opendartreader to determine which version is compatible with other requirements. This could take a while.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
# 영업이익 데이터를 추가
import OpenDartReader
from datetime import datetime
import os
from dotenv import load_dotenv

In [13]:
# .env 파일에서 환경 변수 로드
load_dotenv()

True

In [14]:
api_key = os.getenv('api_key')

In [15]:
df.head(1)

,Close,Volume,MA20,target,MLP_Sentiment
Date,,,,,
2024-05-20 00:00:00+09:00,187236.4375,3592128,NaN,0,0.178482


In [16]:
# 시차 정보를 제거 -> 시차 정보가 없는 시계열 데이터와 시차 정보가 존재하는 시계열 데이터가 결합 x
df.index = df.index.tz_localize(None)
df.head(1)

,Close,Volume,MA20,target,MLP_Sentiment
Date,,,,,
2024-05-20,187236.4375,3592128,NaN,0,0.178482


In [17]:
# dart에서 데이터를 수집
dart = OpenDartReader(api_key)

In [19]:
# 현재년도 로드 -> 최근 3년간 목록을 생성
current_year = datetime.now().year
years = [current_year -2, current_year -1, current_year]
years

[2024, 2025, 2026]

In [ ]:
# 분기 보고서 codes
report_codes = ['11013','11012','11014','11011']
dart_data_list = []

In [22]:
for year in years:
    for code in report_codes:
        try:
            report = dart.finstate('000660', year, code)
            if report is not None:
                op_profit = report[(report['fs_div'] == "CFS") & (report['account_nm'] == '영업이익')]
                if not op_profit.empty:
                    val = int(op_profit['thstrm_amount'].values[0].replace(',',''))
                    # code 11013-> 1분기 보고서
                    if code == '11013' : d = f'{year}-05-15'
                    elif code == '11012' : d = f'{year}-08-14'
                    elif code == '11014' : d = f'{year}-11-14'
                    else : d = f'{year+1}-03-31'

                    report_data = pd.to_datetime(d)
                    if report_data <= datetime.now():
                        dart_data_list.append({'Date' : report_date, 'Operation_Profit' : val})
        except: continue
    dart_data_list

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

